# K3 - a bounded null, not an absent one  (tests C.13.10)

Reads `g7_rebuilt.npz` from K2. The healthy pairs are every combination of four encoders, so they are not independent observations - the primary interval is a **cluster bootstrap over encoders**, with the naive t-interval shown only to size the difference.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# REQUIRES G0_hub_rebuild TO HAVE PASSED ITS VALIDATION GATE.
# The original hub artifacts were never persisted. G0 rebuilds them
# and refuses to write anything unless the reconstruction reproduces
# the published singular values. If G0 stopped, do not run this.
# ==========================================================
# K3 — turn "no detectable effect" into a BOUNDED null  (report C.13.10).
# No new data. Re-reads the six healthy pairs already measured.
#
# WHY. C.13.10 currently reports: healthy spaces +0.010, n=6, two negative,
# verdict NEUTRAL - and bounds it honestly as "'no detectable effect at this
# sample size' rather than a demonstrated null."
#
# That is the correct thing to say and it is also the softest claim in the
# project, while carrying a load-bearing interpretation: transfer runs on
# the global component, which is why the hub works and why it never reaches
# the ceiling. An examiner who wants to push will push here.
#
# It can be hardened without measuring anything new. The report already
# sets the precedent for exactly this treatment - the encoder-count null is
# reported as "0.27 standard errors, with a 95 per cent interval of -0.038
# to +0.050 - NOT distinguishable from sampling noise." The healthy-pairs
# null deserves the same three numbers: an interval, a minimum detectable
# effect, and an equivalence test against a pre-registered bound.
#
# THE CLUSTERING PROBLEM, which is the reason this cell is not two lines.
# Six pairs is not six independent observations. Pair-cosine < 0.30 leaves
# four healthy encoders, and C(4,2) = 6 - so the six pairs are every
# combination of the SAME four encoders. Each encoder appears in three of
# them. A plain t-interval over six pairs assumes independence that does
# not exist and will be too narrow; the effective sample size is closer to
# four than six. So the primary interval here is a CLUSTER bootstrap that
# resamples ENCODERS and rebuilds the pair set, with the naive t-interval
# reported alongside only to show the size of the difference.
#
# PRE-REGISTERED SMALLEST EFFECT OF INTEREST: 0.050.
# Justification, fixed before looking at the interval: the degenerate
# population gains +0.151, and the interpretive claim is that the healthy
# gain is negligible RELATIVE to that. One third of it is the bound. It is
# also under a tenth of mean hub preservation (0.594), so an effect smaller
# than this could not change any downstream reading.
#
# If the interval falls inside +/- SESOI, the claim upgrades from "no
# detectable effect" to "any effect is smaller than 0.05, which is too
# small to matter here." If it does not, the honest output is the minimum
# detectable effect - which is a better sentence than the current one
# either way, because it says what the design COULD have seen.
# ==========================================================
import os
import numpy as np
from pathlib import Path
from itertools import combinations

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
SESOI = 0.050
SEED = 0
N_BOOT = 20000

# --- adjust to your G7 artifact --------------------------------------
G7_NPZ = DATA_DIR / "g7_rebuilt.npz"   # written by K2, the G7 re-run
# expected keys: 'pair_names'  ['a - b', ...] for all 21 pairs
#                'gain'        per-pair change in kNN overlap
#                'pair_cos'    per-pair WORST pair-cosine (the collapse flag)
# ----------------------------------------------------------------------

z = np.load(G7_NPZ, allow_pickle=True)
pair_names = [str(s) for s in z["pair_names"]]
gain = np.asarray(z["gain"], dtype=np.float64)
pair_cos = np.asarray(z["pair_cos"], dtype=np.float64)
assert len(pair_names) == len(gain) == len(pair_cos) == 21

healthy = pair_cos < 0.30
g = gain[healthy]
hn = [pair_names[i] for i in range(21) if healthy[i]]

# recover the encoder membership of each healthy pair
enc_of = [tuple(s.strip() for s in n.split("-")[:2]) if "-" in n else (n, n)
          for n in hn]
encoders = sorted({e for pair in enc_of for e in pair})

print(f"healthy pairs: {len(g)}   encoders involved: {len(encoders)}")
for n, v in zip(hn, g):
    print(f"   {n:<34} {v:+.4f}")
print(f"\nmean {g.mean():+.4f}   sd {g.std(ddof=1):.4f}   "
      f"negative {int((g < 0).sum())} of {len(g)}")
print(f"encoders: {', '.join(encoders)}")
if len(g) == 6 and len(encoders) == 4:
    print("  -> confirmed: 6 pairs from 4 encoders, every combination.")
    print("     Effective n is nearer 4 than 6. Cluster bootstrap is primary.")

In [ ]:
# ---------- naive paired t interval (reported for contrast only) ----------
from math import sqrt
n = len(g)
se = g.std(ddof=1) / sqrt(n)
# t critical, 95%, df = n-1, without scipy
_T = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447,
      7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228}
tcrit = _T.get(n - 1, 1.96)
t_lo, t_hi = g.mean() - tcrit * se, g.mean() + tcrit * se
t_stat = g.mean() / se if se > 0 else 0.0

In [ ]:
# ---------- cluster bootstrap over ENCODERS (primary) ----------
rng = np.random.default_rng(SEED)
idx_of_pair = {frozenset(p): i for i, p in enumerate(enc_of)}
boot = []
for _ in range(N_BOOT):
    draw = rng.choice(encoders, size=len(encoders), replace=True)
    vals = []
    for a, b in combinations(range(len(draw)), 2):
        key = frozenset((draw[a], draw[b]))
        if len(key) == 2 and key in idx_of_pair:      # skip self-pairs
            vals.append(g[idx_of_pair[key]])
    if vals:
        boot.append(np.mean(vals))
boot = np.array(boot)
c_lo, c_hi = np.percentile(boot, [2.5, 97.5])

In [ ]:
# ---------- minimum detectable effect, 80% power, two-sided 5% ----------
mde = (tcrit + 0.842) * g.std(ddof=1) / sqrt(n)

print("\n" + "=" * 64)
print("C.13.10 healthy-pairs null, bounded")
print("=" * 64)
print(f"  observed mean gain                    {g.mean():+.4f}")
print(f"  standard errors from zero             {t_stat:.2f}")
print(f"  naive t 95% interval (n={n}, ignores clustering)")
print(f"                                        {t_lo:+.4f} to {t_hi:+.4f}")
print(f"  CLUSTER bootstrap 95% interval (resamples {len(encoders)} encoders)")
print(f"                                        {c_lo:+.4f} to {c_hi:+.4f}")
print(f"  widening from clustering              "
      f"{((c_hi - c_lo) / (t_hi - t_lo)):.2f}x")
print(f"  minimum detectable effect (80% power) {mde:+.4f}")
print(f"  pre-registered SESOI                  +/-{SESOI:.3f}")

print("\n" + "=" * 64)
inside = (c_lo > -SESOI) and (c_hi < SESOI)
if inside:
    print("VERDICT: EQUIVALENCE DEMONSTRATED at the pre-registered bound.")
    print("The claim upgrades. Write it as: 'on healthy spaces the hub's")
    print(f"effect on local agreement is bounded within +/-{SESOI:.2f}")
    print(f"(cluster-bootstrap 95% CI {c_lo:+.3f} to {c_hi:+.3f}), which is")
    print("under a third of the degenerate-pair gain and cannot change any")
    print("downstream reading.' That is a demonstrated null, not an absence")
    print("of evidence.")
elif mde <= SESOI:
    print("VERDICT: not equivalent, but the design was ADEQUATE - it could")
    print(f"have detected {mde:+.3f}, inside the SESOI. So the honest line is")
    print("'no effect large enough to matter was found, and the design could")
    print("have found one.' Still stronger than the current wording.")
else:
    print("VERDICT: UNDERPOWERED. The design could only have detected")
    print(f"{mde:+.3f}, which is larger than the {SESOI:.3f} bound that would")
    print("matter. Keep the current honest wording and ADD the number:")
    print(f"'six pairs from four encoders can only detect effects above")
    print(f"{mde:.3f}; this is a bound on the design, not evidence of a null.'")
    print("Do not upgrade the claim. Saying exactly what the design could")
    print("not see is a stronger position than leaving it unstated.")

print("\nWhichever lands, report the CLUSTER interval, not the t interval,")
print("and say why in one clause: six pairs are all combinations of four")
print("encoders, so they are not six independent observations. Volunteering")
print("that is much better than having it pointed out.")